Load the transformed data

Import Libraries


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

Start Spark

In [8]:
spark = (
    SparkSession.builder
    .appName("SupportIQ-Joins")
    .master("local[*]")
    .getOrCreate()
)

Define the path

In [9]:
transformed_path = "../data/processed/support_tickets_transformed"

Find the Parquet files

In [10]:
parquet_files = [

    os.path.join(
        transformed_path,
        file
    )

    for file in os.listdir(
        transformed_path
    )

    if file.endswith(".parquet")

]

Load the data

In [11]:
tickets_df = spark.read.parquet(
    *parquet_files
)

print(
    "Transformed data loaded successfully."
)

Transformed data loaded successfully.


In [12]:
print(
    "Rows:",
    tickets_df.count()
)

print(
    "Columns:",
    len(tickets_df.columns)
)

Rows: 1204092
Columns: 25


Create a customer DataFrame

In [13]:
customer_data = [
    ("CUST00001", "Asia Pacific", "Enterprise"),
    ("CUST00002", "Europe", "Standard"),
    ("CUST00003", "North America", "Enterprise"),
    ("CUST00004", "Latin America", "Standard"),
    ("CUST00005", "Europe", "Enterprise")
]

In [14]:
customer_df = spark.createDataFrame(
    customer_data,
    [
        "customer_id",
        "customer_region",
        "customer_type"
    ]
)

In [15]:
customer_df.show()

+-----------+---------------+-------------+
|customer_id|customer_region|customer_type|
+-----------+---------------+-------------+
|  CUST00001|   Asia Pacific|   Enterprise|
|  CUST00002|         Europe|     Standard|
|  CUST00003|  North America|   Enterprise|
|  CUST00004|  Latin America|     Standard|
|  CUST00005|         Europe|   Enterprise|
+-----------+---------------+-------------+



INNER JOIN

In [16]:
inner_join_df = tickets_df.join(
    customer_df,
    tickets_df.customer_id == customer_df.customer_id,
    "inner"
)

In [17]:
inner_join_df.select(
    tickets_df.ticket_id,
    tickets_df.customer_id,
    customer_df.customer_type
).show(
    20,
    truncate=False
)

+----------+-----------+-------------+
|ticket_id |customer_id|customer_type|
+----------+-----------+-------------+
|TKT1130588|CUST00001  |Enterprise   |
|TKT1002672|CUST00001  |Enterprise   |
|TKT0777851|CUST00001  |Enterprise   |
|TKT0587060|CUST00001  |Enterprise   |
|TKT1011212|CUST00001  |Enterprise   |
|TKT0815348|CUST00001  |Enterprise   |
|TKT0360486|CUST00001  |Enterprise   |
|TKT0268670|CUST00001  |Enterprise   |
|TKT1155235|CUST00001  |Enterprise   |
|TKT0679823|CUST00001  |Enterprise   |
|TKT0994375|CUST00001  |Enterprise   |
|TKT0739483|CUST00001  |Enterprise   |
|TKT0586134|CUST00001  |Enterprise   |
|TKT0249366|CUST00001  |Enterprise   |
|TKT0051977|CUST00001  |Enterprise   |
|TKT0517233|CUST00001  |Enterprise   |
|TKT0009715|CUST00001  |Enterprise   |
|TKT0936301|CUST00001  |Enterprise   |
|TKT0663380|CUST00001  |Enterprise   |
|TKT0238848|CUST00001  |Enterprise   |
+----------+-----------+-------------+
only showing top 20 rows


LEFT JOIN

In [18]:
left_join_df = tickets_df.join(
    customer_df,
    tickets_df.customer_id == customer_df.customer_id,
    "left"
)

In [19]:
left_join_df.select(
    tickets_df.ticket_id,
    tickets_df.customer_id,
    customer_df.customer_type
).show(
    20,
    truncate=False
)

+----------+-----------+-------------+
|ticket_id |customer_id|customer_type|
+----------+-----------+-------------+
|TKT0002985|CUST32151  |NULL         |
|TKT0003274|CUST23069  |NULL         |
|TKT0003569|CUST13679  |NULL         |
|TKT0005425|CUST29911  |NULL         |
|TKT0003444|CUST22495  |NULL         |
|TKT0004969|CUST19120  |NULL         |
|TKT0003731|CUST06661  |NULL         |
|TKT0000127|CUST48563  |NULL         |
|TKT0000764|CUST01754  |NULL         |
|TKT0005011|CUST39515  |NULL         |
|TKT0002379|CUST47850  |NULL         |
|TKT0004109|CUST25178  |NULL         |
|TKT0005005|CUST38192  |NULL         |
|TKT0003396|CUST39634  |NULL         |
|TKT0002641|CUST02697  |NULL         |
|TKT0005263|CUST47261  |NULL         |
|TKT0001871|CUST18157  |NULL         |
|TKT0002017|CUST20223  |NULL         |
|TKT0003837|CUST29033  |NULL         |
|TKT0000173|CUST18535  |NULL         |
+----------+-----------+-------------+
only showing top 20 rows


Measure unmatched customers

In [20]:
unmatched_customers = left_join_df.filter(
    F.col("customer_type").isNull()
)

In [21]:
print(
    "Tickets without customer information:",
    unmatched_customers.count()
)

Tickets without customer information: 1203968


LEFT ANTI JOIN

In [22]:
unmatched_tickets = tickets_df.join(
    customer_df,
    tickets_df.customer_id == customer_df.customer_id,
    "left_anti"
)

In [23]:
unmatched_tickets.select(
    "ticket_id",
    "customer_id"
).show(
    20,
    truncate=False
)

+----------+-----------+
|ticket_id |customer_id|
+----------+-----------+
|TKT0057503|CUST21898  |
|TKT0067011|CUST02025  |
|TKT0102847|CUST24038  |
|TKT0105000|CUST30490  |
|TKT0021482|CUST09648  |
|TKT0056287|CUST03281  |
|TKT0059883|CUST21208  |
|TKT0024906|CUST18691  |
|TKT0053797|CUST33744  |
|TKT0046023|CUST06379  |
|TKT0062371|CUST00865  |
|TKT0074022|CUST07002  |
|TKT0057169|CUST03301  |
|TKT0060316|CUST02357  |
|TKT0152453|CUST34491  |
|TKT0043126|CUST35940  |
|TKT0043458|CUST07607  |
|TKT0050137|CUST34854  |
|TKT0100460|CUST41767  |
|TKT0143237|CUST48226  |
+----------+-----------+
only showing top 20 rows


Create an Agent Lookup Table


In [24]:
agent_data = [
    ("AG0001", "Agent A", "Core HR"),
    ("AG0002", "Agent B", "Benefits"),
    ("AG0003", "Agent C", "Payroll"),
    ("AG0004", "Agent D", "Time Management"),
    ("AG0005", "Agent E", "Recruitment")
]

In [25]:
agent_df = spark.createDataFrame(
    agent_data,
    [
        "agent_id",
        "agent_name",
        "team"
    ]
)

In [26]:
agent_df.show()

+--------+----------+---------------+
|agent_id|agent_name|           team|
+--------+----------+---------------+
|  AG0001|   Agent A|        Core HR|
|  AG0002|   Agent B|       Benefits|
|  AG0003|   Agent C|        Payroll|
|  AG0004|   Agent D|Time Management|
|  AG0005|   Agent E|    Recruitment|
+--------+----------+---------------+



Join Tickets with Agents

In [27]:
agent_joined_df = tickets_df.join(
    agent_df,
    tickets_df.agent_id == agent_df.agent_id,
    "left"
).select(
    tickets_df["*"],
    agent_df["agent_name"],
    agent_df["team"]
)

In [28]:
agent_joined_df.select(
    "ticket_id",
    "agent_id",
    "agent_name",
    "team"
).show(
    20,
    truncate=False
)

+----------+--------+----------+--------+
|ticket_id |agent_id|agent_name|team    |
+----------+--------+----------+--------+
|TKT0003731|AG0782  |NULL      |NULL    |
|TKT0004109|AG0051  |NULL      |NULL    |
|TKT0005005|AG0664  |NULL      |NULL    |
|TKT0003444|AG0984  |NULL      |NULL    |
|TKT0002985|AG0025  |NULL      |NULL    |
|TKT0002874|AG0002  |Agent B   |Benefits|
|TKT0003837|AG0742  |NULL      |NULL    |
|TKT0005011|AG0528  |NULL      |NULL    |
|TKT0000764|AG0455  |NULL      |NULL    |
|TKT0001871|AG0627  |NULL      |NULL    |
|TKT0000173|AG0692  |NULL      |NULL    |
|TKT0003569|AG0903  |NULL      |NULL    |
|TKT0002641|AG0925  |NULL      |NULL    |
|TKT0005263|AG0029  |NULL      |NULL    |
|TKT0000127|AG0975  |NULL      |NULL    |
|TKT0002379|AG0323  |NULL      |NULL    |
|TKT0005425|AG0168  |NULL      |NULL    |
|TKT0002017|AG0619  |NULL      |NULL    |
|TKT0004969|AG0119  |NULL      |NULL    |
|TKT0003396|AG0448  |NULL      |NULL    |
+----------+--------+----------+--

Combine both joins

In [29]:
support_enriched_df = tickets_df.join(
    customer_df,
    tickets_df.customer_id == customer_df.customer_id,
    "left"
).select(
    tickets_df["*"],
    customer_df["customer_type"]
)

In [30]:
support_enriched_df = support_enriched_df.join(
    agent_df,
    support_enriched_df.agent_id == agent_df.agent_id,
    "left"
).select(
    support_enriched_df["*"],
    agent_df["agent_name"],
    agent_df["team"]
)

Check row counts after joins

In [31]:
tickets_count = tickets_df.count()

print(
    "Tickets before join:",
    tickets_count
)

Tickets before join: 1204092


In [32]:
enriched_count = support_enriched_df.count()

print(
    "Tickets after join:",
    enriched_count
)

Tickets after join: 1204092


Final join validation

In [33]:
print(
    "Original ticket count:",
    tickets_df.count()
)

print(
    "Enriched ticket count:",
    support_enriched_df.count()
)

Original ticket count: 1204092
Enriched ticket count: 1204092


In [34]:
if tickets_df.count() == support_enriched_df.count():

    print(
        "Join validation passed: row count unchanged."
    )

else:

    print(
        "Warning: row count changed after join."
    ) 

Join validation passed: row count unchanged.
